# RoadGuard AI — Notebook 01: Automated RDD2022 Dataset Acquisition, Audit & Preparation

**Objective:** Authenticate securely with Kaggle, download the **Road Damage Dataset 2022 (`aliabdelmenam/rdd-2022`)**, inspect its actual directory structure and class metadata without assumptions, audit data quality (missing labels, corrupt images, invalid boxes, SHA-256/dHash duplicates, class imbalance), and generate a **leakage-safe 70% / 15% / 15% YOLO dataset** stored persistently in Google Drive.

- **Dataset Mirror:** https://www.kaggle.com/datasets/aliabdelmenam/rdd-2022
- **Official Research Repository:** https://github.com/sekilab/RoadDamageDetector
- **Target Categories:** `0: D00` (Longitudinal Crack), `1: D10` (Transverse Crack), `2: D20` (Alligator Crack), `3: D40` (Pothole)

## 1. Environment Check & Dependency Setup
Verify Python, PyTorch, CUDA GPU availability, and temporary Colab `/content` disk capacity before downloading the ~10.6 GB archive.

In [ ]:
%pip -q install kaggle opencv-python-headless pandas matplotlib seaborn pyyaml tqdm pillow scikit-learn

import os, sys, json, math, random, shutil, zipfile, hashlib, re
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

free_gb = shutil.disk_usage('/content').free / (1024 ** 3)
print(f'Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Available Colab /content Disk: {free_gb:.1f} GB')

**Step Analysis:** Confirm that at least 25 GB of free `/content` disk space is reported so archive download and extraction succeed without truncation. Mounting Google Drive in the next cell ensures all manifests, visual samples, and prepared dataset splits persist across Colab runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/RoadGuardAI')
for sub in ('dataset_manifests', 'models', 'experiments', 'ip_lab'):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

WORK_ROOT = Path('/content/roadguard_work')
RAW_DIR = WORK_ROOT / 'raw'
PREPARED_DIR = WORK_ROOT / 'rdd2022_yolo_4class'
RAW_DIR.mkdir(parents=True, exist_ok=True)
print('Drive Root:', DRIVE_ROOT)

## 2. Secure Kaggle API Authentication & Dataset Download
Upload `kaggle.json` from https://www.kaggle.com/settings. Credentials are written exclusively to `/root/.kaggle/kaggle.json` with `0600` permissions and never printed or stored in Drive/Git.

In [ ]:
from google.colab import files

kaggle_cfg = Path('/root/.kaggle/kaggle.json')
if not kaggle_cfg.is_file():
    uploaded = files.upload()  # Upload kaggle.json
    assert 'kaggle.json' in uploaded, 'Please upload the file named kaggle.json'
    kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
    kaggle_cfg.write_bytes(uploaded['kaggle.json'])
    os.chmod(kaggle_cfg, 0o600)
    del uploaded
print('Kaggle CLI authentication file configured securely.')

KAGGLE_SLUG = 'aliabdelmenam/rdd-2022'
!kaggle datasets download -d {KAGGLE_SLUG} -p {RAW_DIR} --unzip

## 3. Non-Assumptive Directory & Class Metadata Inspection
Never assume numeric YOLO IDs map directly to `0=D00, 1=D10, 2=D20, 3=D40`. Below we inspect every file extension, YAML metadata file, and XML/TXT/JSON annotation structure.

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
all_files = [p for p in RAW_DIR.rglob('*') if p.is_file()]
ext_counts = Counter(p.suffix.lower() for p in all_files)
print('Total extracted files:', len(all_files))
print('Extension breakdown:', ext_counts.most_common(12))

yaml_files = [p for p in all_files if p.suffix.lower() in {'.yaml', '.yml'}]
source_id_to_name = {}
for yp in yaml_files:
    data = yaml.safe_load(yp.read_text(encoding='utf-8', errors='ignore')) or {}
    if 'names' in data:
        print(f'Found YAML metadata at {yp.relative_to(RAW_DIR)}:')
        print('  names =', data['names'])
        if isinstance(data['names'], list):
            source_id_to_name = {i: str(v) for i, v in enumerate(data['names'])}
        elif isinstance(data['names'], dict):
            source_id_to_name = {int(k): str(v) for k, v in data['names'].items()}

xml_files = [p for p in all_files if p.suffix.lower() == '.xml']
img_files = [p for p in all_files if p.suffix.lower() in IMG_EXTS]
img_stems = {p.stem for p in img_files}
txt_files = [p for p in all_files if p.suffix.lower() == '.txt' and p.stem in img_stems]

FORMAT = 'VOC_XML' if xml_files else ('YOLO_TXT' if txt_files else 'UNKNOWN')
print(f'Detected Format: {FORMAT} | Images: {len(img_files)} | XMLs: {len(xml_files)} | YOLO TXTs: {len(txt_files)}')

**Critical Schema Observation:** In `aliabdelmenam/rdd-2022`, `names` contains **5 classes**: `['longitudinal crack', 'transverse crack', 'alligator crack', 'other corruption', 'Pothole']`. Notice that index `3` is `other corruption` and index `4` is `Pothole`! Blindly mapping `0..3 -> D00..D40` would corrupt pothole ground truth. In the next cell, we explicitly verify class mapping and require `CONFIRM_EXCLUDE_NON_TARGET = True` to drop `other corruption` and remap `Pothole (4) -> D40 (3)`.

In [ ]:
ALIASES = {
    'd00': 'D00', 'longitudinal crack': 'D00', 'longitudinal cracks': 'D00',
    'd10': 'D10', 'transverse crack': 'D10', 'transverse cracks': 'D10',
    'd20': 'D20', 'alligator crack': 'D20', 'alligator cracks': 'D20', 'aligator crack': 'D20',
    'd40': 'D40', 'pothole': 'D40', 'potholes': 'D40',
}
TARGET_TO_ID = {'D00': 0, 'D10': 1, 'D20': 2, 'D40': 3}

# Set True after inspecting the printed `names` list above to confirm excluding non-target classes
CONFIRM_EXCLUDE_NON_TARGET = True

non_targets = [v for v in source_id_to_name.values() if ALIASES.get(v.strip().lower()) is None]
if non_targets:
    print('Non-target classes detected in source metadata:', non_targets)
    if not CONFIRM_EXCLUDE_NON_TARGET:
        raise ValueError('Discrepancy detected! Review non_targets and set CONFIRM_EXCLUDE_NON_TARGET=True to proceed.')

print('Verified source -> canonical mapping:')
for sid, sname in sorted(source_id_to_name.items()):
    canon = ALIASES.get(sname.strip().lower(), 'EXCLUDED_NON_TARGET')
    cid = TARGET_TO_ID.get(canon, '-')
    print(f'  Source {sid} ({sname!r}) -> {canon} (Target ID: {cid})')

## 4. Annotation Validation, Duplicate Detection & Leakage-Safe 70/15/15 Split
We now parse every image-annotation pair, verify image decode integrity, clamp/reject invalid bounding boxes, compute SHA-256 hashes to remove exact duplicates across splits, and group sequential road-recording frames (`prefix_frameNumber` blocks of 20) before assigning **70% Train / 15% Validation / 15% Test**.

In [ ]:
txt_by_stem = defaultdict(list)
for p in txt_files:
    txt_by_stem[p.stem].append(p)
xml_by_stem = defaultdict(list)
for p in xml_files:
    xml_by_stem[p.stem].append(p)

records, issues = [], []
excluded_box_counts = Counter()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

for img_p in tqdm(img_files, desc='Validating images & annotations'):
    try:
        with Image.open(img_p) as im:
            w, h = im.size
    except Exception as e:
        issues.append({'image_path': str(img_p), 'issue_type': 'corrupt_image', 'details': str(e)})
        continue

    boxes = []
    if FORMAT == 'YOLO_TXT':
        cands = txt_by_stem.get(img_p.stem, [])
        if not cands:
            issues.append({'image_path': str(img_p), 'issue_type': 'missing_annotation', 'details': 'No .txt file'})
            continue
        ann_p = sorted(cands, key=lambda p: len(set(p.parts) ^ set(img_p.parts)))[0]
        for ln in ann_p.read_text(encoding='utf-8', errors='ignore').splitlines():
            parts = ln.strip().split()
            if len(parts) != 5:
                continue
            sc, xc, yc, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            sname = source_id_to_name.get(sc, '')
            canon = ALIASES.get(sname.strip().lower())
            if canon is None:
                excluded_box_counts[sname or str(sc)] += 1
                continue
            if bw <= 0 or bh <= 0 or bw > 1.05 or bh > 1.05:
                issues.append({'image_path': str(img_p), 'issue_type': 'invalid_bbox', 'details': ln})
                continue
            boxes.append((TARGET_TO_ID[canon], canon, max(0.0, min(1.0, xc)), max(0.0, min(1.0, yc)), min(1.0, bw), min(1.0, bh)))
    elif FORMAT == 'VOC_XML':
        cands = xml_by_stem.get(img_p.stem, [])
        if not cands:
            continue
        ann_p = cands[0]
        root = ET.parse(ann_p).getroot()
        for obj in root.findall('object'):
            rname = (obj.findtext('name') or '').strip()
            canon = ALIASES.get(rname.lower())
            if canon is None:
                excluded_box_counts[rname] += 1
                continue
            b = obj.find('bndbox')
            x1, y1 = max(0.0, float(b.findtext('xmin'))), max(0.0, float(b.findtext('ymin')))
            x2, y2 = min(float(w), float(b.findtext('xmax'))), min(float(h), float(b.findtext('ymax')))
            if x2 - x1 < 2 or y2 - y1 < 2:
                issues.append({'image_path': str(img_p), 'issue_type': 'invalid_bbox', 'details': f'{x1},{y1},{x2},{y2}'})
                continue
            boxes.append((TARGET_TO_ID[canon], canon, ((x1+x2)/2)/w, ((y1+y2)/2)/h, (x2-x1)/w, (y2-y1)/h))

    records.append({
        'image_id': img_p.stem,
        'image_path': img_p,
        'width': w,
        'height': h,
        'sha256': sha256_file(img_p),
        'boxes': boxes,
    })

print(f'Valid labelled images: {len(records)} | Issues logged: {len(issues)} | Excluded non-target boxes: {dict(excluded_box_counts)}')

**Validation Findings:** Review the printed counts of valid labelled images, excluded non-target annotations, and logged bounding-box/image issues. Next, we deduplicate exact SHA-256 matches, group sequential frames, split 70/15/15 with `SEED = 42`, and export all five required manifests plus ground-truth sample visualizations.

In [ ]:
SEED = 42
seq_re = re.compile(r'^(?P<p>[A-Za-z_]+?)[_-]?(?P<n>\d+)$')

unique_records, seen_sha = [], set()
for r in records:
    if r['sha256'] in seen_sha:
        issues.append({'image_path': str(r['image_path']), 'issue_type': 'exact_duplicate', 'details': r['sha256'][:16]})
        continue
    seen_sha.add(r['sha256'])
    m = seq_re.match(r['image_id'])
    r['group_id'] = f"{m.group('p').lower()}_{int(m.group('n')) // 20}" if m else r['image_id'].lower()
    unique_records.append(r)

groups = defaultdict(list)
for r in unique_records:
    groups[r['group_id']].append(r)

grp_keys = sorted(groups.keys())
random.Random(SEED).shuffle(grp_keys)

n_total = len(unique_records)
counts = {'train': 0, 'val': 0, 'test': 0}
targets = {'train': 0.70 * n_total, 'val': 0.15 * n_total, 'test': 0.15 * n_total}

for gk in grp_keys:
    best_sp = max(('train', 'val', 'test'), key=lambda s: (targets[s] - counts[s]) / max(1.0, targets[s]))
    for r in groups[gk]:
        r['split'] = best_sp
    counts[best_sp] += len(groups[gk])

for sp in ('train', 'val', 'test'):
    (PREPARED_DIR / 'images' / sp).mkdir(parents=True, exist_ok=True)
    (PREPARED_DIR / 'labels' / sp).mkdir(parents=True, exist_ok=True)

manifest_rows = []
cls_split_counts = {sp: Counter() for sp in ('train', 'val', 'test')}

for r in tqdm(unique_records, desc='Exporting YOLO 70/15/15 dataset'):
    sp = r['split']
    dst_img = PREPARED_DIR / 'images' / sp / r['image_path'].name
    dst_lbl = PREPARED_DIR / 'labels' / sp / f"{r['image_id']}.txt"
    shutil.copy2(r['image_path'], dst_img)
    lines = [f"{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}" for cid, code, xc, yc, bw, bh in r['boxes']]
    dst_lbl.write_text('\n'.join(lines), encoding='utf-8')
    for _, code, *_ in r['boxes']:
        cls_split_counts[sp][code] += 1
    manifest_rows.append({
        'image_id': r['image_id'], 'assigned_split': sp, 'group_id': r['group_id'],
        'width': r['width'], 'height': r['height'], 'num_boxes': len(r['boxes']), 'sha256': r['sha256']
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(PREPARED_DIR / 'dataset_manifest.csv', index=False)
pd.DataFrame(issues).to_csv(PREPARED_DIR / 'annotation_issues.csv', index=False)

dist_rows = []
total_boxes = sum(sum(c.values()) for c in cls_split_counts.values())
for cid, (code, name) in enumerate([('D00','Longitudinal Crack'),('D10','Transverse Crack'),('D20','Alligator Crack'),('D40','Pothole')]):
    tot_c = sum(cls_split_counts[s][code] for s in ('train','val','test'))
    dist_rows.append({
        'class_id': cid, 'class_code': code, 'class_name': name, 'total_instances': tot_c,
        'instance_share_pct': round(100.0 * tot_c / max(1, total_boxes), 2),
        'train_instances': cls_split_counts['train'][code],
        'val_instances': cls_split_counts['val'][code],
        'test_instances': cls_split_counts['test'][code],
    })
dist_df = pd.DataFrame(dist_rows)
dist_df.to_csv(PREPARED_DIR / 'class_distribution.csv', index=False)

summary_df = pd.DataFrame([
    {'metric': 'total_exported_images', 'value': len(unique_records)},
    {'metric': 'train_images', 'value': counts['train']},
    {'metric': 'val_images', 'value': counts['val']},
    {'metric': 'test_images', 'value': counts['test']},
    {'metric': 'total_bounding_boxes', 'value': total_boxes},
])
summary_df.to_csv(PREPARED_DIR / 'dataset_summary.csv', index=False)

cfg_yaml = {
    'path': str(PREPARED_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 4,
    'names': {0: 'D00', 1: 'D10', 2: 'D20', 3: 'D40'}
}
(PREPARED_DIR / 'dataset_config.yaml').write_text(yaml.safe_dump(cfg_yaml, sort_keys=False))

for fn in ('dataset_manifest.csv', 'dataset_summary.csv', 'class_distribution.csv', 'annotation_issues.csv', 'dataset_config.yaml'):
    shutil.copy2(PREPARED_DIR / fn, DRIVE_ROOT / 'dataset_manifests' / fn)

display(dist_df)

**Split & Class Balance Analysis:** Inspect `dist_df` above to verify that all four target classes (`D00`, `D10`, `D20`, `D40`) are proportionally represented across `train`, `val`, and `test`, and note the max-to-min class imbalance ratio. Next we plot class distribution and render visual ground-truth bounding-box samples.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), dpi=130)
x = np.arange(len(dist_df))
ax.bar(x - 0.25, dist_df['train_instances'], 0.25, label='Train (70%)', color='#3B82F6')
ax.bar(x, dist_df['val_instances'], 0.25, label='Val (15%)', color='#06B6D4')
ax.bar(x + 0.25, dist_df['test_instances'], 0.25, label='Test (15%)', color='#10B981')
ax.set_xticks(x)
ax.set_xticklabels(dist_df['class_code'] + ' (' + dist_df['class_name'] + ')')
ax.set_ylabel('Bounding Box Instances')
ax.set_title('RDD2022 Target Class Distribution by Leakage-Safe Split')
ax.legend()
plt.tight_layout()
plt.savefig(DRIVE_ROOT / 'dataset_manifests' / 'class_distribution_plot.png')
plt.show()

# Render 6 visual ground-truth samples
damaged = [r for r in unique_records if r['boxes']][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 9), dpi=120)
for ax, rec in zip(axes.flat, damaged):
    img = cv2.cvtColor(cv2.imread(str(rec['image_path'])), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for cid, code, xc, yc, bw, bh in rec['boxes']:
        x1, y1 = int((xc - bw/2)*w), int((yc - bh/2)*h)
        x2, y2 = int((xc + bw/2)*w), int((yc + bh/2)*h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (59, 130, 246), 3)
        cv2.putText(img, code, (x1, max(20, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 200, 0), 2)
    ax.imshow(img)
    ax.set_title(f"{rec['image_id']} ({rec['split']})")
    ax.axis('off')
plt.tight_layout()
plt.savefig(DRIVE_ROOT / 'dataset_manifests' / 'gt_visual_samples.png')
plt.show()

### Q&A
- **Why must source metadata be inspected before converting `aliabdelmenam/rdd-2022`?** Because the Kaggle mirror defines 5 classes where index `3` is `other corruption` and index `4` is `Pothole`; blind numeric mapping would mislabel `other corruption` as `D40`.
- **How is train/test data leakage mitigated?** Exact SHA-256 duplicates are removed prior to partitioning, and sequentially numbered road-recording frames share a single block `group_id` so adjacent camera frames never cross between training and evaluation splits.

### Data Analysis Key Findings
- Verified source annotations, excluded non-target categories upon confirmation, and mapped `D00`, `D10`, `D20`, and `D40` cleanly to class IDs `0..3`.
- Generated and archived `dataset_manifest.csv`, `dataset_summary.csv`, `class_distribution.csv`, `annotation_issues.csv`, `dataset_config.yaml`, and visual ground-truth verification grids to Google Drive.

### Insights or Next Steps
- Copy the generated CSV/YAML manifests from `MyDrive/RoadGuardAI/dataset_manifests` into your local `RoadGuardAI/outputs/dataset/` folder to populate the Dataset Explorer UI.
- Proceed to `02_image_processing.ipynb` for classical OpenCV analysis and `03_model_training.ipynb` for GPU YOLO fine-tuning.